# Call peaks and generate sample embeddings

This notebook describes how to call peaks with hotspot3 and turn them into a motif embedding of the sample - the cell-type input of the vinson model.<br>
_Use case: to make predictions for a new DNase-seq or ATAC-seq sample, call peaks and calculate its embedding._<br>
**Input:** aligned reads (BAM) of the sample<br>
**Output:** peak calls and a 637-dimensional motif embedding of the sample<br>
**Requirements:** `hotspot3`, `tabix` (htslib) and `bedops` (`sort-bed`, `bedmap`) on PATH.

## Imports

In [1]:
import numpy as np
import pandas as pd

# Peak calling

To call peaks in DNase I and ATAC data, we use the developed peak caller hotspot3 [github.com/vierstralab/hotspot3](https://github.com/vierstralab/hotspot3). It is tailored to work on datasets without a control experiment (ATAC-seq and DNase-seq) using adaptive estimation of the background (nonspecific cleavages) with a negative binomial distribution. hotspot3 accounts for variation in both total signal level and signal-to-background ratio along the genome.

In [2]:
!hotspot3 --help

usage: hotspot3 [-h] [--bam BAM] [--chrom_sizes CHROM_SIZES]
                [--fdrs FDRS [FDRS ...]] [--reference REFERENCE] [--cpus CPUS]
                [--outdir OUTDIR] [--debug] [--tempdir TEMPDIR]
                [--mappable_bases MAPPABLE_BASES] [--window WINDOW]
                [--background_window BACKGROUND_WINDOW]
                [--signal_quantile SIGNAL_QUANTILE] [--cutcounts CUTCOUNTS]
                [--signal_parquet SIGNAL_PARQUET]
                [--pvals_parquet PVALS_PARQUET] [--fdrs_parquet FDRS_PARQUET]
                [--chromosomes CHROMOSOMES [CHROMOSOMES ...]] [--save_density]
                id

Call peaks using hotspot3 with local background estimation and MODWT smoothing.

positional arguments:
  id                    Unique identifier of the sample

optional arguments:
  -h, --help            show this help message and exit
  --bam BAM             Path to input bam/cram file
  --chrom_sizes CHROM_SIZES
                        Path to chromosome sizes file

## To obtain hotspot3 peak calls run
```bash
hotspot3 <sample_id> \
    --bam <bam> \
    --chrom_sizes hg38.chrom.sizes \
    --mappable_bases mappable_bases_file.bed.gz \
    --fdrs 0.01 0.05 \
    --cpus 6 \
    --debug 
    
```

### README and description of the output files: [github.com/vierstralab/hotspot3](https://github.com/vierstralab/hotspot3)

# Generating sample embeddings from peak calls

hotspot3 writes peak calls at 1% FDR (`<sample_id>.peaks.fdr0.01.bed.gz`). The embedding is calculated from them in 5 steps:
1) Replace each peak with a 150 bp window centred on its summit (summit −75 … +74 bp) to control for variable width
2) Overlap the windows with the motif scans
3) For each motif, compute the proportion of windows overlapping at least one hit
4) Median across motifs belonging to the same non-redundant TF cluster (method described in [Global reference mapping of human transcription factor footprints](https://www.nature.com/articles/s41586-020-2528-x))
5) Log-transform and z-score across clusters

![embeddings](figures/Embeddings.png)

## Step 1: 150 bp summit windows
Each peak is replaced by `[summit - 75, summit + 75)`, i.e. summit −75 … +74 bp (summit is column 7 of the hotspot3 peak file). Intermediate files are written to `outputs/`.
### Note: the example is restricted to chr22 to keep it fast; use `zcat $P` instead of `tabix $P chr22` to process the whole sample.

In [3]:
%%bash
cd outputs

P=https://regulome.altius.org/data/per_sample/AG95417/peaks/fdr0.01/AG95417.peaks.fdr0.01.bed.gz

# 150 bp windows centred on the peak summit (column 7)
tabix $P chr22 | awk -v OFS="\t" '{
    start = $7 - 75; if (start < 0) start = 0;
    print $1, start, $7 + 75, $4
}' | sort-bed - > AG95417.150bp.bed

wc -l AG95417.150bp.bed

    1740 AG95384.150bp.bed


## Step 2: overlap with motif hits
A window overlaps a motif if it shares ≥1 bp with any hit of that motif.
### Note: overlapping with remote motif scans is ~40% slower. To process many samples, download the scans file locally.

In [4]:
%%bash
cd outputs

M=https://resources.altius.org/~sabramov/collaborations/vinson/examples/human.moods_all.bed.gz
#M=https://resources.altius.org/~sabramov/collaborations/vinson/examples/mouse.moods_all.bed.gz # mouse (mm10) samples
#M=/net/seq/data2/projects/sabramov/ENCODE4/motif_scans.p1e_minus_5/human/human.moods_all.bed.gz

time tabix $M -R AG95417.150bp.bed \
    | sort-bed - \
    | bedmap --echo-map-id-uniq --delim ';' \
        AG95417.150bp.bed - > AG95417.motif_hits.txt

real	0m52.860s


user	0m2.562s


sys	0m0.814s


In [5]:
# one line per window: ';'-separated motif ids, empty if no hit
motif_overlap = pd.read_table('outputs/AG95417.motif_hits.txt', names=['motifs'], skip_blank_lines=False)
n_windows = len(motif_overlap)
n_windows

1740

## Step 3: motif proportions in 150 bp windows
Fraction of windows with at least one hit, for every motif (motifs without hits get 0).

In [6]:
motif_metadata_path = 'https://resources.altius.org/~sabramov/collaborations/vinson/examples/motifs_meta.tsv'

motif_metadata = pd.read_table(motif_metadata_path, index_col=0)
motif_metadata

,cluster,source_id,tf_name,family_name,motif_type,PMID,moods_scans,pfm,motif_hits_path
motif_id,,,,,,,,,
BANP_Grand2021,AC0375,BANP,BANP,Unknown,ChIP-seq,34234345,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
M00217_2.00,AC0237,ZNF200_H322Y,ZNF200,C2H2 ZF,PBM,27013732,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
M00218_2.00,AC0623,ZNF200_REF,ZNF200,C2H2 ZF,PBM,27013732,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
M00219_2.00,AC0623,ZNF200_S265Y,ZNF200,C2H2 ZF,PBM,27013732,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
M00220_2.00,AC0606,SNAI2_D119E,SNAI2,C2H2 ZF,PBM,27013732,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
...,...,...,...,...,...,...,...,...,...
MA1984.1,AC0490,MA1984.1,ZNF667,['C2H2 zinc finger factors'],Unknown,27852650,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
MA1985.1,AC0406,MA1985.1,ZNF669,['C2H2 zinc finger factors'],Unknown,27852650,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
MA1986.1,AC0124,MA1986.1,ZNF692,['C2H2 zinc finger factors'],Unknown,25690854,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...


In [7]:
n_windows_with_hit = motif_overlap['motifs'].dropna().str.split(';').explode(
).str.replace('.pfm', '', regex=False).value_counts()

motif_proportions = n_windows_with_hit.reindex(
    motif_metadata.index, fill_value=0
) / n_windows
motif_proportions

motif_id
BANP_Grand2021    0.002874
M00217_2.00       0.001724
M00218_2.00       0.007471
M00219_2.00       0.006897
M00220_2.00       0.019540
                    ...   
MA1984.1          0.001724
MA1985.1          0.009195
MA1986.1          0.035057
MA1987.1          0.061494
MA2003.1          0.003448
Name: count, Length: 5193, dtype: float64

## Step 4: median within TF cluster

In [8]:
motif_proportions_by_cluster = motif_proportions.groupby(
    motif_metadata['cluster']
).median().rename('AG95417').to_frame().T
motif_proportions_by_cluster

cluster,AC0001,AC0002,AC0003,AC0004,AC0005,AC0006,AC0007,AC0008,AC0009,AC0010,...,AC0628,AC0629,AC0630,AC0631,AC0632,AC0633,AC0634,AC0635,AC0636,AC0637
AG95384,0.0,0.001149,0.0,0.004598,0.002299,0.001724,0.0,0.000575,0.001149,0.000575,...,0.012644,0.047701,0.016667,0.002299,0.002299,0.002299,0.006897,0.004023,0.006322,0.005172


## Step 5: log z-score normalization
`log(proportion + 1e-5)`, then z-scored across the 637 clusters of the sample — same as the embeddings used to train the cell-type classifier.

In [9]:
def log_zscore_normalization(x, eta=1e-5):
    x = np.log(x + eta)
    return (x - x.mean()) / x.std()

In [10]:
motif_embedding = motif_proportions_by_cluster.apply(
    log_zscore_normalization, eta=1e-5, axis=1
)
motif_embedding

cluster,AC0001,AC0002,AC0003,AC0004,AC0005,AC0006,AC0007,AC0008,AC0009,AC0010,...,AC0628,AC0629,AC0630,AC0631,AC0632,AC0633,AC0634,AC0635,AC0636,AC0637
AG95384,-3.086401,-0.70723,-3.086401,-0.016566,-0.362437,-0.505715,-3.086401,-1.049888,-0.70723,-1.049888,...,0.489101,1.153442,0.627284,-0.362437,-0.362437,-0.362437,0.186028,-0.083251,0.14254,0.042269


### Save the embedding (used in 2.Predict_density.ipynb)

In [11]:
motif_embedding.to_csv('outputs/AG95417.motif_embedding.tsv', sep='\t')

**Next:** predict density with this embedding - 2.Predict_density.ipynb